In [1]:
import nfstream
import numpy as np
import os
import pandas as pd
import pefile
import seaborn as sns
import struct
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import re

le = LabelEncoder()

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import TransformerEncoder, TransformerEncoderLayer


### network data feature extraction

In [ ]:
dataset_dir = '/home/mani/Desktop/main_project/data/network_data/ids-dataset/'
benign_dir = os.path.join(dataset_dir, 'benign/')
malicious_dir = os.path.join(dataset_dir, 'malicious/')
benign_files = os.listdir(benign_dir)
malicious_files = os.listdir(malicious_dir)

features = ['src_port', 'dst_port', 'protocol', 'ip_version', 'vlan_id',
	   'tunnel_id', 'bidirectional_first_seen_ms',
	   'bidirectional_last_seen_ms', 'bidirectional_duration_ms',
	   'bidirectional_packets', 'bidirectional_bytes', 'src2dst_first_seen_ms',
	   'src2dst_last_seen_ms', 'src2dst_duration_ms', 'src2dst_packets',
	   'src2dst_bytes', 'dst2src_first_seen_ms', 'dst2src_last_seen_ms',
	   'dst2src_duration_ms', 'dst2src_packets', 'dst2src_bytes',
	   'application_name', 'application_category_name', 'user_agent',
	   'content_type']

def get_features_from_pacp_file(pcap_file):
	flow = nfstream.NFStreamer(source=pcap_file, statistical_analysis=False).to_pandas()
	# encode the application_name, application_category_name, user_agent, content_type
	flow['application_name'] = le.fit_transform(flow['application_name'])
	flow['application_category_name'] = le.fit_transform(flow['application_category_name'])
	flow['user_agent'] = le.fit_transform(flow['user_agent'])
	flow['content_type'] = le.fit_transform(flow['content_type'])
	# drop the columns which are not required
	flow = flow.drop(['id', 'expiration_id', 'src_ip', 'src_mac', 'src_oui', 'dst_ip', 'dst_mac', 'dst_oui', 'application_is_guessed', 'application_confidence', 'requested_server_name', 'client_fingerprint', 'server_fingerprint'], axis=1)
	flow = flow.drop(["vlan_id", "tunnel_id"], axis=1)
	flow = flow.fillna(0)
	flow = flow.drop_duplicates()
	# print the column name if it has string datatype
	for col in flow.columns:
		if flow[col].dtype == 'object':
			print(col)
	features = flow.values.tolist()
	return features

# Create a DataFrame for benign features
benign_features = []
for file in tqdm(benign_files):
	features = get_features_from_pacp_file(benign_dir + file)
	benign_features.extend(features)

# Create a DataFrame for malicious features
malicious_features = []
for file in tqdm(malicious_files):
	features = get_features_from_pacp_file(malicious_dir + file)
	malicious_features.extend(features)

benign_df = pd.DataFrame(benign_features, columns=features)
malicious_df = pd.DataFrame(malicious_features, columns=features)
# add label column
benign_df['label'] = 0
malicious_df['label'] = 1
# concat benign and malicious dataframe
network_data = pd.concat([benign_df, malicious_df])

network_data.to_csv('../data/csv/network_data.csv', index=False)

### portable excutable file feature extraction

In [7]:
def get_features_from_pe(file_path):
    # Extract header information
    PEfile = pefile.PE(file_path)
    feature_vector = [PEfile.OPTIONAL_HEADER.DATA_DIRECTORY[6].Size, PEfile.OPTIONAL_HEADER.DATA_DIRECTORY[6].VirtualAddress, PEfile.OPTIONAL_HEADER.MajorImageVersion, PEfile.OPTIONAL_HEADER.MajorOperatingSystemVersion, PEfile.OPTIONAL_HEADER.DATA_DIRECTORY[0].VirtualAddress,
                      PEfile.OPTIONAL_HEADER.DATA_DIRECTORY[0].Size, PEfile.OPTIONAL_HEADER.DATA_DIRECTORY[12].VirtualAddress, PEfile.OPTIONAL_HEADER.DATA_DIRECTORY[2].Size, PEfile.OPTIONAL_HEADER.MajorLinkerVersion, PEfile.FILE_HEADER.NumberOfSections, PEfile.OPTIONAL_HEADER.SizeOfStackReserve, PEfile.OPTIONAL_HEADER.DllCharacteristics]

    machine_type = PEfile.FILE_HEADER.Machine
    num_sections = len(PEfile.sections)
    entry_point = PEfile.OPTIONAL_HEADER.AddressOfEntryPoint
    feature_vector.extend([machine_type, num_sections,  entry_point])

    # Extract section information
    section_info = [section.SizeOfRawData for section in PEfile.sections]
    feature_vector += section_info

    # Extract ASCII and Unicode strings
    with open(file_path, 'rb') as f:
        data = f.read()
    ascii_strings = re.findall(b'[ -~]{4,}', data)
    unicode_strings = re.findall(b'[\x20-\x7E\x80-\xFE]{4,}', data)
    feature_vector.extend([len(ascii_strings), len(unicode_strings)])

    # Extract import information
    imports = {}
    for entry in PEfile.DIRECTORY_ENTRY_IMPORT:
        try:
            dll_name = entry.dll.decode('utf-8')
            imports[dll_name] = [func.name.decode(
                'utf-8') for func in entry.imports]
        except:
            dll_name = entry.dll
    feature_vector.extend(list(imports.keys()))

    return feature_vector


benign_dir = "/home/mani/Desktop/main_project/data/DikeDataset/files/benign"
malicious_dir = "/home/mani/Desktop/main_project/data/DikeDataset/files/malware"


pe_data = pd.DataFrame(columns=['Name', 'MachineType', 'NumberOfSections', 'AddressOfEntryPoint',
                  'SizeOfRawData', 'NumberOfAsciiStrings', 'NumberOfUnicodeStrings', 'Imports', 'Label'])

for file_name in tqdm(os.listdir(benign_dir)):
    file_path = os.path.join(benign_dir, file_name)
    try:
        features = get_features_from_pe(file_path)
        pe_data.loc[len(pe_data.index)] = [file_name, features[0], features[1],
                                 features[2], features[3], features[4], features[5], features[6], 0]
    except Exception as e:
        # print(f"file {file_name} caused error")
        # print(e)
        continue

for file_name in tqdm(os.listdir(malicious_dir)):
    file_path = os.path.join(malicious_dir, file_name)
    try:
        features = get_features_from_pe(file_path)
        pe_data.loc[len(pe_data.index)] = [file_name, features[0], features[1],
                                 features[2], features[3], features[4], features[5], features[6], 1]
    except Exception as e:
        # print(f"file {file_name} caused error")
        # print(e)
        continue

# label encoding
pe_data['Imports'] = le.fit_transform(pe_data['Imports'])

# save data
pe_data.to_csv("../data/csv/pe_data.csv", index=False)

100%|██████████| 10841/10841 [07:27<00:00, 24.25it/s]


### data loading

In [2]:
network_data = pd.read_csv("../data/csv/network_data.csv")
pe_data = pd.read_csv("../data/csv/pe_data.csv")

# train test split
from sklearn.model_selection import train_test_split
x_network = network_data.drop(['label'], axis=1)
y_network = network_data['label']
x_pe = pe_data.drop(['Name', 'Label'], axis=1)
y_pe = pe_data['Label']

x_network_train, x_network_test, y_network_train, y_network_test = train_test_split(
    x_network, y_network, test_size=0.2, random_state=42)
x_pe_train, x_pe_test, y_pe_train, y_pe_test = train_test_split(	x_pe, y_pe, test_size=0.2, random_state=42)


### model 